# Durability emulator — 1. Dataset generation

Stage 1 of the split durability pipeline. This notebook **only** loads the carbonation model, runs
the emulator, and saves the resulting lambda datasets — no PCE is fitted here. Stage 2, in
[`02_train_pce.ipynb`](02_train_pce.ipynb), only reads what this notebook writes.

Full context and the original single-notebook version of this pipeline are in
[`../pipeline_final_test.ipynb`](../pipeline_final_test.ipynb).

Functions come from [`functions_final.py`](../functions_final.py), one directory up:
`generate_dataset_at_time_durability` calls `emulator_function_time_durability` twice per time step
(once on the training design points, once on a fresh validation sample) and saves both.

**Artefacts written per time step**,
`<n_latent_samples>_<kind>_<split>_<t>_install_<year>_cement_<type>_exposure_<exposure>.pkl`:

| kind | split | content |
|---|---|---|
| `dataset_full` | train / val | one row per latent replica: inputs, effective values, carbonation depth, $g$, lambdas, processing time |
| `dataset_unique` | train / val | one row per design point: inputs, the four lambdas, processing time |

Plus one aggregate `<n_latent_samples>_emulator_timing_durability.pkl`, read back by stage 2 to
compute the emulator/surrogate speed-up.

## 1. Libraries

In [ ]:
import sys
import time
from pathlib import Path

# functions_final.py sits one directory up, in beam_problem_1/
sys.path.insert(0, str(Path.cwd().parent))

import dill
import joblib
import numpy as np
import pandas as pd

from functions_final import *
from UQpy.distributions import Uniform, JointIndependent

## 2. Load the carbonation model

In [ ]:
name_best_model = Path.cwd().parent / 'model_NeuralNetwork_MLP_fold_4.pkl'

model = joblib.load(name_best_model)
print("Carbonation model loaded successfully!")
print(f"   Expected features: {model.feature_names_in_}")

## 3. Random variables and fixed parameters

Design variables (compressive strength, relative humidity, cover) plus everything the emulator
needs that isn't a design variable.

In [ ]:
fck_min = 20  # MPa
fck_max = 50  # MPa
rh_min  = 20  # %
rh_max  = 80  # %
cov_min = 15  # mm
cov_max = 60  # mm

cement_type          = 3
installation_year    = 1990
exposure_conditions  = 2
n_samples            = 1000     # Number of design samples
n_latent_samples     = 100000   # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 250      # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)

## 4. Design samples

In [ ]:
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist = Uniform(loc=cov_min, scale=cov_max - cov_min)
joint    = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

x_pce_rvs = joint.rvs(n_samples)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")
print("\nSample statistics:")
print(f"   fck:   {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:    {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover: {x_pce_rvs[:, 2].min():.1f} - {x_pce_rvs[:, 2].max():.1f} mm (mean: {x_pce_rvs[:, 2].mean():.1f} mm)")

## 5. Time grid

In [ ]:
times = np.linspace(0, 150, 10, endpoint=True)  # Time points for carbonation depth prediction
# times = [10, 20, 30, 40]
times

## 6. Generate the dataset at each time step

A fresh validation sample is drawn for every time step (matching the original combined pipeline).
`generate_dataset_at_time_durability` does the rest: latent sampling, carbonation depth prediction,
GLD fit, and saving `dataset_full`/`dataset_unique` for both splits.

In [ ]:
print("="*60)
print("GENERATING THE DURABILITY DATASET")
print("="*60)

generation_results = []
for t in times:
    x_val = joint.rvs(n_samples_validation)
    result = generate_dataset_at_time_durability(
                                                    x_train=x_pce_rvs,
                                                    x_val=x_val,
                                                    carb_model=model,
                                                    time_step=t,
                                                    cement_type=cement_type,
                                                    installation_year=installation_year,
                                                    exposure_conditions=exposure_conditions,
                                                    n_latent_samples=n_latent_samples,
                                                    output_dir='.',
                                                )
    generation_results.append(result)

## 7. Timing summary

Cost of building the dataset, per time step. `Train total (s)` is what stage 2 will compare against
the PCE's own evaluation time to compute the speed-up.

In [ ]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

with open(f'{n_latent_samples}_emulator_timing_durability.pkl', 'wb') as f:
    dill.dump(emulator_timing, f)

print(f"Total emulator time over the whole grid: {emulator_timing['Train total (s)'].sum() + emulator_timing['Val total (s)'].sum():.1f} s")
emulator_timing